# SAFS check — rebuild the San Andreas deck from raw data, end to end

This is the **real-data** counterpart to `deck_workflow.ipynb`. It runs the whole chain on
the San Andreas fault system using the actual community models, and compares what it builds
against a deck we already run on Frontera.

Everything lands in its own folder: **`outputs/safs_check/`**. Nothing here touches the
demo, and nothing overwrites a shipped deck.

### Before you start

```bash
python tools/stage_safs_data.py          # ~129 MB, gitignored
```

That collects the raw products from the legacy tree into `data/safs_alt/`:

| staged as | what it is | size |
|:--|:--|:--|
| `raw/CSM_orientation.csv` | Yang & Hauksson YHSM-2013 community stress model | 8.6 MB |
| `raw/cvm/CVM_*_h_data.csv` | 37 slices of the statewide "muscal" CVM (Vp, Vs, ρ), 0–70 km | 25 MB |
| `raw/ctm/CTM_*_h_data_final.csv` | 105 slices of the SCEC Community Thermal Model, 0–21 km | 36 MB |
| `mesh_alt.puml.h5` | the ALT fault mesh | 59 MB |

**Runtime: roughly 4–6 minutes**, dominated by the two Delaunay interpolations (the CVM is
37 slices onto a 452 × 361 grid; the CTM is 105 slices). This is not the 5-minute demo —
it is the real thing.

### How to trace anything in this notebook

Every code cell opens with a `══ STEP ══` header listing the functions it calls and the
file each one lives in, so you can go from a printed number to the code that produced it
without searching.

| you want | look at |
|:--|:--|
| what a stage does and its gates | `deckbuild/<stage>.py` — `material`, `stress`, `friction`, `mesh`, `deck` |
| the descriptor schema and validation | `deckbuild/config.py` |
| the fault frame, snapping, tractions | `deckbuild/geometry.py` |
| the NetCDF layout SeisSol reads | `deckbuild/asagi.py` |
| raw slices → a uniform grid | `deckbuild/rawslices.py` |
| the mesh↔deck compatibility check | `deckbuild/stage_f.py` |
| what a gate name means | the `GateReport` line prints the gate id; grep it in `deckbuild/` |
| the module map and conventions | `CODEBASE_GUIDE.md` |
| meshing | `MESHING.md`, then `skills/code-mesh-build-improve/SKILL.md` |
| why the design is shaped this way | `docs/PLAN_integrated_workflow_notebook_2026-08-01.md` |

In Jupyter you can also jump straight to a source file:

```python
from deckbuild.material import MaterialStage
MaterialStage.build??          # shows the source and its path
```

Every gate id (`M1`, `V2`, `G4`, `F0`, `A`–`G`, `P1`–`P8`) is a literal string in the
module that raises it, so `grep -rn '"P5"' deckbuild/` takes you to the check itself.

## [0] Setup

In [ ]:
# ══ STEP 0 · load the SAFS descriptor against the staged raw data ══════════════
#   tools/stage_safs_data.py  put the raw products where this descriptor expects them
#   init()                    deckbuild/bootstrap.py
#   Project.load()            deckbuild/config.py   require_files=True -> every input
#                                                   must exist, or it raises
#   projects/safs_alt.yaml    the descriptor: grids, strike frame, hypocentre, readers
# ---- PARAMETERS ----------------------------------------------------------------
PROJECT      = "safs_alt"
RAW_DATA_DIR = None     # None -> data/safs_alt/ (what stage_safs_data.py fills)
OUT_NAME     = "safs_check"
# Compare against a shipped deck?  Point this at one, or None to skip the comparison.
SHIPPED_DECK = ("~/Downloads/seisol_quakeworx/"
                "safs_seisol_v4_0_0_RSSRW_ALT_THERMAL_CASE1_intermediate"
                "_plast_phi30_40_gradedfw_k1p70_nwredM7p8_sefw0_attenuation_deep40km")
# ----------------------------------------------------------------------------------
import time
from pathlib import Path
import numpy as np
from deckbuild.bootstrap import init
from deckbuild.config import Project

B = init(project=PROJECT, require_files=False)
kw = {} if RAW_DATA_DIR is None else {"data_dir": Path(RAW_DATA_DIR).expanduser()}
cfg = Project.load(B.root / "projects" / f"{PROJECT}.yaml", require_files=True, **kw)
OUT = B.root / "outputs" / OUT_NAME
DECK = B.root / "decks" / f"{PROJECT}_check"
for sub in ("material", "stress", "friction"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)
shipped = Path(SHIPPED_DECK).expanduser() if SHIPPED_DECK else None
if shipped and not shipped.is_dir():
    print(f"  ! shipped deck not found at {shipped}; comparisons will be skipped")
    shipped = None
print(cfg.summary()); print(f"outputs      : {OUT}")

## [1] What the shipped deck used

Before rebuilding anything, read the design straight out of the deck we are targeting. The
introspector recovers it from the files themselves — including three values that exist
**only** as literals inside emitted Lua.

In [ ]:
# ══ STEP 1 · read the SHIPPED deck's design out of its own files ═══════════════
#   introspect_deck()  deckbuild/introspect.py
#     nc axes            -> the four grids
#     easi !ConstantMap  -> rs_b, rs_sl0
#     the rs_muw Lua     -> the strike frame AND the f_w design (they exist ONLY there)
#     the Tnuc_s Lua     -> the hypocentre, radius, amplitude
#     bulkFriction field -> phi (NOT the yaml prose, which has been wrong)
#     parameters.par     -> Plasticity, Tv, the attenuation band
#   Anything it cannot infer is marked UNKNOWN, never guessed.
from deckbuild.introspect import introspect_deck
facts = introspect_deck(shipped) if shipped else {}
if facts:
    for k in ("k_from_filename", "freeze_from_filename", "fw_values",
              "fw_boundaries_s_km", "hypocenter_xyz", "nucleation_radius_m",
              "nucleation_amplitude", "phi_deg_from_field", "rs_b_constant",
              "strike_azimuth_deg", "strike_origin_xy", "Plasticity", "FreqCentral"):
        if k in facts:
            print(f"  {k:22} = {facts[k]}")
    print(f"\n  {len(facts)} fields recovered, {len(facts.unknown)} unknown")

## [2] Mesh — ingest and gate

The ALT fault mesh, at production scale. This ingests and gates it; it does not build it —
see `MESHING.md`.

In [ ]:
# ══ STEP 2 · ingest the ALT mesh and snap the hypocentre ═══════════════════════
#   MeshStage.build()  deckbuild/mesh.py      register the .puml.h5
#   load_fault()       deckbuild/geometry.py  BC-3 facets + the Tandem basis
#   snap_hypocenter()  deckbuild/geometry.py  HARD-fails past snap_tol_m
# This is production scale -- expect ~30 s just to read the mesh.
from deckbuild.mesh import MeshStage
from deckbuild.geometry import load_fault, snap_hypocenter
t0 = time.time()
mesh_art = MeshStage().build(cfg, OUT)
fault = load_fault(cfg.mesh(), cfg.data_dir, strike=cfg.strike)
snap = snap_hypocenter(cfg.hypocenter(), fault, cfg.strike, cfg.crs,
                       gate_bands=cfg.gate_bands)
print(f"{len(fault):,} fault facets in {time.time()-t0:.0f}s")
snap.report.print()

## [3] Material — 37 raw CVM slices → nc, + Sv(z)

The two-stage recipe: reproject each slice, interpolate onto the inscribed UTM grid, form
the moduli **at the source nodes**, then resample in z.

Plasticity is opt-in here too — the shipped deck used φ = 30/40, which is **not** the
Roten paper's 35/45.

In [ ]:
# ══ STEP 3 · MATERIAL from 37 raw CVM slices, + 105 CTM slices ═════════════════
#   MaterialStage.build()   deckbuild/material.py
#     _cvm_slices()         deckbuild/material.py  -> rawslices.interp_slices()
#     _ctm_slices()         deckbuild/material.py  -> the thermal nc
#     interp_slices()       deckbuild/rawslices.py  reproject -> INSCRIBED grid ->
#                             LinearNDInterpolator per slice -> depth-to-elevation ->
#                             moduli AT THE SOURCE NODES -> z-resample
#     derive_plasticity()   deckbuild/material.py  Roten 2014; phi 30/40 here, NOT 35/45
#   The two interpolations dominate the runtime (~40 s CVM, ~115 s CTM).
# ---- PARAMETERS ----------------------------------------------------------------
WITH_PLASTICITY  = True     # the shipped deck has Plasticity = 1
PHI_SOFT, PHI_HARD = 30.0, 40.0   # the SAFS production values, NOT Roten's 35/45
WITH_ATTENUATION = True     # the shipped deck is a viscoelastic run
# ----------------------------------------------------------------------------------
from deckbuild.material import AttenuationSpec, MaterialStage, PlasticitySpec
t0 = time.time()
mstage = MaterialStage()
mat = mstage.build(cfg, OUT / "material",
                   plasticity=PlasticitySpec(PHI_SOFT, PHI_HARD) if WITH_PLASTICITY else None,
                   attenuation=AttenuationSpec() if WITH_ATTENUATION else None)
print(f"material + thermal built in {time.time()-t0:.0f}s")
mstage.verify(cfg, mat, plasticity_artifact=mat.plasticity).print()

In [ ]:
# ══ STEP 3b · compare the rebuilt CVM against the shipped nc ═══════════════════
#   asagi_axes(), read_asagi()  deckbuild/asagi.py
#   The two-tier comparison harness is tests/nc_compare.py (compare_nc); this cell does
#   the same thing inline so the numbers are visible.
#   OPEN QUESTION: the grid reproduces exactly, the VALUES do not.  See
#   docs/EXERCISE_safs_reproduction.md.
# Compare the rebuilt CVM against the shipped one.
from deckbuild.asagi import asagi_axes, read_asagi
if shipped:
    s_nc = next(shipped.glob("*material_cvm.nc"), None)
    if s_nc:
        gx, gy, gz = asagi_axes(mat.material.path)
        sx, sy, sz = asagi_axes(s_nc)
        print(f"  shape   built {len(gx)}x{len(gy)}x{len(gz)}   "
              f"shipped {len(sx)}x{len(sy)}x{len(sz)}")
        for n, a, b in (("x", gx, sx), ("y", gy, sy), ("z", gz, sz)):
            print(f"  {n} axis identical: {np.array_equal(a, b)}")
        if (len(gx), len(gy), len(gz)) == (len(sx), len(sy), len(sz)):
            _, _, _, bf, _ = read_asagi(mat.material.path, fields=["mu", "rho"])
            _, _, _, sf, _ = read_asagi(s_nc, fields=["mu", "rho"])
            d = np.abs(bf["mu"] - sf["mu"])
            rel = d / np.maximum(np.abs(sf["mu"]), 1e-30)
            print(f"  mu   max|d| {d.max():.4g} Pa  max rel {rel.max():.3e}  "
                  f"median rel {np.median(rel):.3e}")
            print(f"  rho  exactly equal on {np.mean(bf['rho']==sf['rho'])*100:.2f}% of nodes")
            print("\n  NOTE: the grid reproduces exactly; the VALUES do not yet.")
            print("  The difference is LOCALISED (docs/EXERCISE_safs_reproduction.md):")
            print("   - a fixed ~2.5% interior stripe at every depth = degenerate Delaunay")
            print("     (agrees exactly AT the sample points, differs between them);")
            print("   - one anomalous level, z = -250 m: the only shallow output level")
            print("     that is NOT itself a CVM slice depth -> the vertical resample.")
            print("  Still OPEN; the workflow does not claim reproduction until it closes.")

### What the rebuilt velocity model looks like

In [ ]:
# ══ STEP 3c · LOOK at the rebuilt material, with the real SAF on it ════════════
#   plot_material_slices()  deckbuild/plots.py
#     fault_trace()         deckbuild/geometry.py  the map trace AT each slice depth
#     _section_along_fault() deckbuild/plots.py    samples the nc down the fault's own
#                                                  CURVED trace, x-axis = along-strike s
# The numbers above say the grid reproduces and the values do not.  These panels say
# WHERE: the shallow basins are the part that differs most, and they are also the part
# that controls ground motion, so the disagreement is not cosmetic.
import matplotlib.pyplot as plt
from deckbuild.plots import plot_material_slices
plot_material_slices(mat.material.path, fault, depths_km=(0.0, 5.0, 10.0), cfg=cfg,
                     sv_profile=mat.sv_profile.path)
plt.show()

## [4] Stress — CSM orientation × Sv × k = 1.70

The shipped deck uses a **constant** k = 1.70 and, per its filename (verified against the
field below), **no** shallow freeze.

In [ ]:
# ══ STEP 4 · STRESS at the shipped deck's k = 1.70 ═════════════════════════════
#   KDesign, StressStage.build/verify   deckbuild/stress.py
#   The shipped nc has no _freeze tag -> the shallow freeze is OFF (verified against the
#   FIELD in the Phase 8 ladder, exercise_safs_reproduction.py rung E2c).
# ---- PARAMETERS ----------------------------------------------------------------
K_VALUES       = [1.70]     # the shipped deck's closure ratio
K_BOUNDARIES   = []
FREEZE_ABOVE_M = 0.0        # the shipped nc has NO _freeze tag -> the freeze is OFF
# ----------------------------------------------------------------------------------
from deckbuild.stress import KDesign, StressStage
t0 = time.time()
sstage = StressStage()
kdes = KDesign(k_values=tuple(K_VALUES),
               boundaries_s_km=tuple(tuple(b) for b in K_BOUNDARIES))
stress = sstage.build(cfg, OUT / "stress", sv_profile=mat.sv_profile, design=kdes,
                      freeze_above_depth_m=FREEZE_ABOVE_M)
print(f"stress built in {time.time()-t0:.0f}s")
sstage.verify(cfg, stress, design=kdes).print()

### On-fault stress on the real fault

In [ ]:
# ══ STEP 4b · project the stress onto the REAL SAF and look at it ══════════════
#   project_stress_onto_fault()  deckbuild/stage_f.py  samples the WRITTEN nc at the
#                                                      160,280 real facet centroids
#   plot_onfault()               deckbuild/plots.py    (s, depth) panels + a map view;
#                                                      derived dtau_dyn and S panels
# The map view is the one to check first: its kinks ARE the mesh's kinks, so a trace that
# looks wrong here means the mesh, not the stress.
from deckbuild.stage_f import project_stress_onto_fault
from deckbuild.plots import plot_onfault
from deckbuild.friction import FwDesign
import matplotlib.pyplot as plt, numpy as np
sn, tau, mu = project_stress_onto_fault(stress.path, fault)
plot_onfault(fault, cfg, {"sigma_n (MPa)": sn, "tau_0 (MPa)": tau, "mu_app": mu},
             snap=snap, f0=FwDesign.f0)   # 0.6, the dataclass default
plt.show()
print(f"min sigma_n {np.nanmin(sn):.2f} MPa | min tau_0 {np.nanmin(tau):.3f} MPa | "
      f"max mu_app {np.nanmax(mu):.3f}")

## [5] Friction — CTM temperature, CASE 1, and the 7-region graded f_w

The shipped deck is CASE 1 (a velocity-strengthening shallow lid) with a seven-region
`f_w` design — recovered in section [1] straight out of its Lua.

In [ ]:
# ══ STEP 5 · FRICTION: CASE 1 + the shipped 7-region f_w ═══════════════════════
#   FrictionStage.build()   deckbuild/friction.py  from the thermal nc built in STEP 3
#   FwDesign / build_fw_map / verify_fw_map  deckbuild/friction.py
#   FW_VALUES defaults to what STEP 1's introspector recovered from the shipped Lua.
# ---- PARAMETERS ----------------------------------------------------------------
CASE = 1     # CASE 1 = VS shallow / VW seismogenic / VS deep (see deck_workflow.ipynb)
# Take the f_w design from the shipped deck when we have it, else the 7-region default.
FW_VALUES     = facts.get("fw_values", [0.0, 0.0, 0.045, 0.03, 0.06, 0.0175, 0.05])
FW_BOUNDARIES = facts.get("fw_boundaries_s_km",
                          [(20, 28), (150, 160), (176, 182), (188, 194),
                           (214, 226), (244, 252)])
F0 = 0.6     # the rate-and-state reference friction; f_w must satisfy 0 <= f_w < f0
# ----------------------------------------------------------------------------------
from deckbuild.friction import FrictionStage, FwDesign, nucleation_lua
t0 = time.time()
fstage = FrictionStage()
friction = fstage.build(cfg, OUT / "friction", case=CASE, thermal=mat.thermal)
fstage.verify(cfg, friction).print()

fwdes = FwDesign(fw_values=tuple(float(v) for v in FW_VALUES),
                 boundaries_s_km=tuple(tuple(map(float, b)) for b in FW_BOUNDARIES),
                 f0=F0)
fw = fstage.build_fw_map(cfg, OUT / "friction", fwdes)
fstage.verify_fw_map(cfg, fw, fwdes).print()
print(f"friction built in {time.time()-t0:.0f}s")

In [ ]:
# ══ STEP 5b · is our LuaMap the SAME FUNCTION as the shipped one? ══════════════
#   _eval_lua_text()  deckbuild/friction.py  re-evaluates an emitted Lua's arithmetic in
#     numpy.  Reads BOTH our form `X + inc` and the legacy `X + inc * g`.
#   This compares the FUNCTIONS, not the text -- formatting may differ.
# Does our emitted rs_muw Lua match the shipped one?
# The shipped maps use the LEGACY form `X + inc * g`; ours writes `X + inc`.  Both are
# read by _eval_lua_text, so this compares the FUNCTIONS, not the text.
if shipped:
    s_lua = next(shipped.glob("*rs_muw*.yaml"), None)
    if s_lua:
        from deckbuild.friction import _eval_lua_text
        ss = np.linspace(-50, 500, 1101)
        try:
            a = _eval_lua_text(Path(fw.path).read_text(), ss)
            b = _eval_lua_text(s_lua.read_text(), ss)
            print(f"  shipped: {s_lua.name}")
            print(f"  f_w(s) max |ours - shipped| over s in [-50, 500] km: "
                  f"{np.max(np.abs(a - b)):.3e}")
            print("  (0 means our emitted LuaMap is the same function as the shipped one)")
        except Exception as exc:
            print(f"  ! could not compare the Lua maps: {exc}")

## [6] Assemble + pre-flight

Into `decks/safs_alt_check/` — its own folder, never a shipped deck.

In [ ]:
# ══ STEP 6 · assemble a CHECK deck and pre-flight it ═══════════════════════════
#   DeckSpec / resolve_paths / DeckStage.assemble / .preflight   deckbuild/deck.py
#   nucleation_lua()                                             deckbuild/friction.py
#   Writes to decks/safs_alt_check/ -- its own folder, never a shipped deck.
from deckbuild.deck import DeckSpec, DeckStage, resolve_paths
arts = {"material": mat.material, "stress": stress, "friction": friction,
        "mesh": mesh_art}
if mat.plasticity is not None:
    arts["plasticity"] = mat.plasticity
spec = DeckSpec(prefix="safs_", plasticity=mat.plasticity is not None,
                attenuation=mat.attenuation, mu_s=0.6, end_time_s=150.0,
                rs_muw_lua=Path(fw.path).read_text(),
                nucleation_lua=nucleation_lua(snap, 2000.0, 75.0e6),
                notes="# Built by safs_check.ipynb -- a CHECK deck, not a production run.")
for k, e in resolve_paths(DECK, arts, spec).items():
    print(f"  {k:12} -> {e['filename']}")
dstage = DeckStage()
deck = dstage.assemble(cfg, DECK, arts, spec, overwrite=True)
dstage.preflight(cfg, deck, spec=spec).print()

## [7] Verdict

In [ ]:
# ══ STEP 7 · verdict ═══════════════════════════════════════════════════════════
# The full E0-E6 ladder, and what is still open, is in
#   exercise_safs_reproduction.py  (the runner)
#   docs/EXERCISE_safs_reproduction.md  (the results)
print(f"deck : {deck}")
for f in sorted(deck.iterdir()):
    print(f"  {f.name:48} {f.stat().st_size/1e6:8.1f} MB")
if shipped:
    print(f"\nshipped deck for comparison: {shipped.name}")
    print("  file-by-file diff is NOT run here: the two decks use different meshes")
    print("  (ALT vs the deep40km variant) and different EndTime/output settings.")
    print("  The meaningful comparisons are the FIELD ones in [3] and [5] above.")
print("\nSee docs/EXERCISE_safs_reproduction.md for the E0-E6 ladder and what is")
print("still open.  Grid exact; CVM values localised to 2 effects, not yet closed.")